# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. All data elements (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. The `@id` uniquely identifies entities within the dataset and must be used for referencing.

We'll print an overview of all record sets and their fields, identified by their `@id`.

In [ ]:
def print_record_sets_overview(ds):
    print("Available Record Sets and Fields (@id):\n")
    for record_set in ds.metadata.record_sets:
        print(f"- RecordSet @id: {record_set.id}")
        print(f"  Name: {record_set.name if hasattr(record_set, 'name') else ''}")
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}, Name: {field.name}")
        print()
# Print the record sets and fields overview
print_record_sets_overview(dataset)

We'll show some sample records for the main record set. Identify the `@id` of the primary record set you'd like to explore (for this dataset, it is likely there is only one main data table—replace as necessary after reviewing the overview above).

_Example below assumes record set ID is `/7154140/d18d681b-431c-4fc3-b535-1e26cb261034` (update this if different from above output)._

In [ ]:
# Set the @id of the main record set here (update if needed based on overview above)
main_record_set_id = '/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'

# Print out a few sample records
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather the list of record set @id's from the dataset metadata
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns and a sample for the main record set
print(f'Columns in the main record set ({main_record_set_id}):')
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All columns are referenced by their `@id` or their canonical column name.

In [ ]:
# Identify a numeric field to analyze (use the column name from the DataFrame; for example, 'Peptide count')
numeric_field = 'Peptide count'
threshold = 10
df = dataframes[main_record_set_id]

# Filter records with Peptide count > threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
normed_col = f"{numeric_field}_normalized"
filtered_df[normed_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, normed_col]].head())

# Group by another field if available
# For example, 'Sample' if there's such a field
group_field = 'Sample'  # Update if a different grouping field exists in your dataset
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing means):")
    print(grouped_df[[numeric_field]].head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using, e.g., matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If there's a group field, show boxplot
if group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset, examined record sets and fields using their `@id`, and performed basic data analysis such as filtering, normalization, grouping, and visualization. Further analysis can be carried out depending on research objectives, including advanced feature engineering, statistical testing, or machine learning modeling.